# 5.1 Concept Drift

In real-world data streams, relationships between input variables and target outcomes often change over time.
**Concept drift** occurs when the statistical properties of the target variable — or its relationship with input features — shift over time.

Common examples include:
- Changing customer preferences in online stores  
- Evolving climate patterns affecting disaster frequencies  
- Economic or policy changes influencing reported damages

If not detected and addressed, concept drift can make previously trained models **less accurate** for future predictions.

In [ ]:
# %pip install geopandas geodatasets

In [ ]:
from pathlib import Path

candidate_paths = [
  Path("configs/visualisations.mplstyle"),
  Path("05_Exercise_5/configs/visualisations.mplstyle"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
  raise FileNotFoundError("Could not find configs/visualisations.mplstyle. Start Jupyter in the repository root or 05_Exercise_5.")

print(f"Loading: {data_path.resolve()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# import custom data handling function from src folder
from src.data_handling import _load_NASA_data
# set custom plotting style
plt.style.use(str(data_path.resolve()))
# import custom visualization function from src folder
from src.visual_helper import defined_colors

## Dataset Overview

In this exercise, we use a NASA disaster dataset that records various disaster events, their characteristics, affected regions, years, and estimated damages in USD.  
Each record typically contains:
- **Location and date** of the event  
- **Disaster type** (e.g., flood, earthquake, cyclone)  
- **Socioeconomic indicators** (e.g., affected population, deaths)  
- **Economic impact** (`Total Damages (US$)`)

This data allows us to explore **temporal patterns** and how the relationships between variables (features and outcomes) may evolve over time.

In [ ]:
df = _load_NASA_data()
df.head()

## **TASK: Investigating Temporal Shifts**
First, try to understand the data and analyze how disaster characteristics change over time by visualizing trends in key metrics. For this purpose, focus on a specific ``Disaster Type`` (e.g., Flood) and examine how ``Total Damages (US$)`` evolves over the years.

Complete the following steps by replacing the ``None`` placeholders with the correct entries:
1. Filter data for a specific disaster type
2. Calculate yearly median and mean damages
3. Plot event count over time
4. Plot median damages over time
5. Boxplot of damages by decade
6. Calculate rolling mean and std deviation, and plot it

In [ ]:
from scipy import stats

# Define variables to focus on
disaster_type = "Flood"
target = "Total Damages (US$)"
time_column = "Year"

In [ ]:
# (1.) Filter and prepare data

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

In [ ]:
# (2.) Calculate yearly median and mean damages

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

In [ ]:
# (3.) Plot event count over time
plt.figure(figsize=(8, 5))

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

plt.show()

In [ ]:
# (4.) Plot median damages over time
plt.figure(figsize=(8, 5))

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

# Labels for plotting
plt.xlabel('Year')
plt.ylabel('Log Median Damages')
plt.title(f'Median {target} Over Time - {disaster_type}')
plt.legend()
plt.show()

In [ ]:
# (5.) Boxplot of damages by decade
plt.figure(figsize=(8, 5))

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

plt.boxplot(data_by_decade, label=decades)
plt.xlabel('Decade')
plt.ylabel('Log Damages')
plt.title(f'Distribution of {target} by Decade - {disaster_type}')
plt.show()

In [ ]:
# (6.) Calculate rolling mean and std deviation, and plot it
# specify window size
window_size = 5

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

# Labels for plotting
plt.xlabel('Year')
plt.ylabel('Log Median Damages')
plt.title(f'Moving Average Analysis - {disaster_type}')
plt.legend()
plt.show()

## **TASK: Investigating Seasonality**
Now that you understand the relationship between floods and damages better, analyze recurring patterns in disaster data by examining how damages vary across different months. This helps identify seasonal patterns that might affect model predictions.

Complete the following steps by replacing the ``None`` placeholders with the correct entries:
1. Calculate statistics for each month
2. Visualize median damages and std. by month and compare to overall median
3. Describe some of the interesting patterns in your own words and propose an explanation for these patterns

In [ ]:
# Define variables to focus on
disaster_type = "Flood"
target = "Total Damages (US$)"
time_column = "Start Month"

# Copy, filter, and prepare data for trend analysis
df_season = df[df["Disaster Type"] == disaster_type].copy()
df_season = df_season[[time_column, target, "Year"]].dropna()
df_season = df_season[df_season[target] > 0]
df_season[target + " (log)"] = np.log1p(df_season[target])

In [ ]:
# (1.) Calculate statistics for each month

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

In [ ]:
# (2.) Visualize median damages and std. by month and compare to overall median
# Set up subplots
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

# Plot 1: Median damages by month
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Log Median Damages')
axes[0].set_title(f'Median {target} by Month - {disaster_type}')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_names, rotation=0)
axes[0].legend(loc='upper right')

# Plot 2: Event count by month
axes[1].bar(monthly_stats['Month'], monthly_stats['Count'], 
               alpha=0.7, color=defined_colors[3])
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of Events')
axes[1].set_title(f'Number of {disaster_type} Events by Month')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names, rotation=0)

# Plot 3: Heatmap Year vs Month
import numpy.ma as ma
pivot_data = df_season.groupby(['Year', time_column])[target + " (log)"].median().reset_index()
pivot_table = pivot_data.pivot(index='Year', columns=time_column, values=target + " (log)")
pivot_filled = pivot_table.fillna(np.nan)  
pivot_array = pivot_filled.to_numpy(dtype=float)
masked_data = ma.masked_invalid(pivot_array)
im = axes[2].imshow(masked_data, aspect='auto', cmap='YlOrRd', 
                        interpolation='nearest')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Year')
axes[2].set_title(f'Heatmap: Log Median {target} - {disaster_type}')
axes[2].set_xticks(range(12))
axes[2].set_xticklabels(month_names, rotation=0)
axes[2].set_yticks(range(0, len(pivot_table), 10))
axes[2].set_yticklabels(pivot_table.index[::10])
plt.colorbar(im, ax=axes[2], label='Log Damages')

plt.subplots_adjust(hspace=0.4)
plt.show()

3. Describe some of the interesting patterns in your own words and propose an explanation for these patterns

...